In [1]:
# 이 노트북에서 사용할 LangChain·OpenAI·실험 패키지를 설치합니다.
# -q는 설치 로그를 줄이고, ==는 수업에서 같은 버전을 사용하게 합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-text-splitters==1.1.2" "python-dotenv==1.2.3"


Note: you may need to restart the kernel to use updated packages.


<h3>개념 요약</h3>
<p>검색 증강 생성(RAG)은 문서를 미리 색인하는 흐름과 질문마다 관련 문서를 찾는 흐름으로 나뉩니다. 이 절에서는 검색 문맥이 없을 때 생성 모델이 사내 규정을 답할 수 없는 이유부터 확인합니다.</p>

In [2]:
# os는 환경 변수를 읽고, dotenv는 현재 작업 폴더에서 .env를 찾습니다.
# env_file에는 찾은 경로가 들어가며, and는 경로 확인 후 로드를 순서대로 실행합니다.
# assert는 파일과 API 키가 없을 때 즉시 이유를 알리고 실행을 멈춥니다.
import os

from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
assert env_file and load_dotenv(env_file), '프로젝트 루트 .env를 찾지 못했습니다.'
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY가 없습니다.'
print('환경 준비 완료')


환경 준비 완료


<h3>예제 실행</h3>
<p>가상 기관의 규정을 검색 문맥 없이 실제 생성 모델에 질문합니다.</p>

In [3]:
# init_chat_model의 provider:model 문자열은 OpenAI 대화 모델을 선택합니다.
# temperature=0은 응답의 무작위성을 낮추고, question은 모델에 전달할 문자열입니다.
# invoke()는 AIMessage를 반환하며 .text로 응답 본문만 읽습니다.
from langchain.chat_models import init_chat_model

model = init_chat_model('openai:gpt-5.4-mini', temperature=0)
question = '가상 기관 파란도서관의 정기 휴관일은 언제인가요?'
reply = model.invoke(question)
print(reply.text)


“파란도서관”의 정기 휴관일 정보가 현재 대화에 제공되어 있지 않아 확인할 수 없습니다.

원하시면 제가 다음 형태로 도와드릴 수 있어요:
- **가상의 운영 규정**을 만들어 정기 휴관일을 설정해 드리기
- **이름/운영시간/휴관일**을 포함한 도서관 안내문 작성하기
- 실제 기관이라면 **공식 홈페이지나 공지문 기준**으로 확인할 수 있도록 안내하기

원하시는 방식으로 말씀해 주세요.


<h3>변형 실행</h3>
<p><code>question</code>을 다른 가상 기관의 내부 규정 질문으로 바꾸어 한 번 더 실행합니다. 모델이 공개 지식과 사내 문서를 구분해 답하는지 확인합니다.</p>

In [4]:
# question에 다른 가상 회사의 사내 규정 질문을 다시 할당합니다.
# 앞 셀의 model을 재사용하고, changed에는 새 AIMessage가 저장됩니다.
question = '가상 회사 해솔상사의 경조 휴가는 며칠인가요?'
changed = model.invoke(question)
print(changed.text)


제가 가진 정보만으로는 **가상 회사 “해솔상사”의 경조 휴가 일수**를 확인할 수는 없습니다.  
해솔상사의 **사내 규정집, 취업규칙, 인사규정**에 따라 달라집니다.

원하시면 제가 바로 도와드릴 수 있어요:
- **경조휴가 규정 예시**를 만들어드리기
- **한국 회사에서 흔한 경조휴가 기준**을 알려드리기
- 해솔상사 **가상 인사규정**을 작성해드리기

원하시는 방식으로 말씀해 주세요.


<h3>실행 검증</h3>
<p>앞 셀의 결과가 이 절에서 확인할 조건을 만족하는지 단언문으로 검사합니다.</p>

In [5]:
# 두 .text가 빈 문자열이 아닌지 and로 함께 검사합니다.
# not in은 검색 문맥을 주지 않은 응답에 문서 출처 형식이 없는지 확인합니다.
assert reply.text and changed.text
assert '[문서 p.' not in changed.text
print('검증 통과: 생성 호출은 성공했고 RAG 문맥은 아직 없습니다.')


검증 통과: 생성 호출은 성공했고 RAG 문맥은 아직 없습니다.


<h3>정리</h3>
<p>생성 모델 호출만으로는 사내 문서의 답과 출처를 보장할 수 없습니다. 다음 절부터 문서를 나누고 검색 가능한 형태로 준비합니다.</p>

<p><strong>원인</strong> — 문서 문맥을 제공하지 않은 응답에서 사내 규정의 출처를 검사하면 실패합니다.</p>

In [6]:
# before_reply는 앞에서 실제로 받은 changed를 가리키므로 API를 다시 호출하지 않습니다.
# try 블록의 출처 assert는 의도적으로 실패하고, except가 AssertionError만 포착합니다.
# 이 구조는 Run All을 중단하지 않으면서 실패 원인을 출력합니다.
before_reply = changed
print(before_reply.text)
try:
    assert '[해솔상사 규정 p.1]' in before_reply.text
except AssertionError:
    print('예상한 실패: 제공된 근거와 출처가 없습니다.')


제가 가진 정보만으로는 **가상 회사 “해솔상사”의 경조 휴가 일수**를 확인할 수는 없습니다.  
해솔상사의 **사내 규정집, 취업규칙, 인사규정**에 따라 달라집니다.

원하시면 제가 바로 도와드릴 수 있어요:
- **경조휴가 규정 예시**를 만들어드리기
- **한국 회사에서 흔한 경조휴가 기준**을 알려드리기
- 해솔상사 **가상 인사규정**을 작성해드리기

원하시는 방식으로 말씀해 주세요.
예상한 실패: 제공된 근거와 출처가 없습니다.


<p><strong>수정</strong> — 질문과 함께 짧은 규정 문장과 출처 태그를 제공합니다.</p>

In [7]:
# evidence는 답과 출처가 함께 있는 최소 검색 문맥입니다.
# f 삼중 문자열은 evidence와 question 값을 여러 줄 프롬프트에 삽입합니다.
# after_reply의 본문에서 휴가 기간과 출처 태그를 각각 검사합니다.
evidence = '[해솔상사 규정 p.1] 본인 결혼 경조 휴가는 5일입니다.'
after_prompt = f"""다음 근거만 사용해 답하고 출처를 붙이세요.
{evidence}
질문: {question}"""
after_reply = model.invoke(after_prompt)
print(after_reply.text)
assert '5일' in after_reply.text
assert '[해솔상사 규정 p.1]' in after_reply.text


가상 회사 해솔상사의 **본인 결혼 경조 휴가**는 **5일**입니다.  
출처: **[해솔상사 규정 p.1] 본인 결혼 경조 휴가는 5일입니다.**
